In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="evaluation",
    notebook_name="04_human_evaluation.ipynb"
)

# Human Evaluation: Agreement Metrics & Pairwise Comparison

This notebook builds on [human-evaluation.md](./human-evaluation.md). There you learned *why* human evaluation matters and what makes it reliable. Here you will implement the key tools from scratch:

1. **Cohen's Kappa** — measure agreement between two annotators, corrected for chance
2. **Annotator disagreement simulation** — see how noisy annotators affect Kappa
3. **IAA visualization** — heatmaps showing where annotators agree and disagree
4. **Pairwise comparison & ELO** — rank models from head-to-head matchups

By the end, you will be able to compute agreement metrics by hand and explain what makes a human evaluation study trustworthy.

---
## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

np.random.seed(42)
print("Setup complete.")

---
## 1. Raw Agreement — Why It Is Not Enough

Two judges rate 100 chatbot responses as "good" or "bad." They agree on 90 out of 100. That sounds great — 90% agreement!

But what if 95 of the 100 responses are obviously good? Two judges who both always say "good" would agree 90.25% of the time by pure chance. The 90% agreement adds almost nothing beyond random guessing.

Let's see this problem in code.

In [ ]:
# Two annotators rate 100 items as good (1) or bad (0)
# Most items are obviously good, so both tend to say "good"

annotator_1 = [1]*80 + [1]*5  + [0]*5  + [0]*10   # 85 good, 15 bad
annotator_2 = [1]*80 + [0]*5  + [1]*5  + [0]*10   # 85 good, 15 bad

# Raw agreement: how often do they give the same label?
agree_count = sum(a == b for a, b in zip(annotator_1, annotator_2))
raw_agreement = agree_count / len(annotator_1)

print(f"Total items: {len(annotator_1)}")
print(f"Times they agree: {agree_count}")
print(f"Raw agreement: {raw_agreement:.0%}")
print()

# Now compute chance agreement
# If both annotators label randomly with their own base rates,
# how often would they agree by accident?
p1_good = sum(annotator_1) / len(annotator_1)  # Annotator 1's rate of saying "good"
p2_good = sum(annotator_2) / len(annotator_2)  # Annotator 2's rate of saying "good"
p1_bad = 1 - p1_good
p2_bad = 1 - p2_good

chance_agreement = (p1_good * p2_good) + (p1_bad * p2_bad)

print(f"Annotator 1 says 'good': {p1_good:.0%} of the time")
print(f"Annotator 2 says 'good': {p2_good:.0%} of the time")
print(f"Chance agreement: {chance_agreement:.1%}")
print()
print(f"So 90% raw agreement vs {chance_agreement:.1%} by chance.")
print("The real agreement beyond chance is much smaller than 90% suggests!")

---
## 2. Cohen's Kappa — From Scratch

Cohen's Kappa fixes the raw agreement problem. It asks: "How much of the agreement is *beyond* what we would expect by chance?"

The formula is:
```
κ = (observed agreement - chance agreement) / (1 - chance agreement)
```

- κ = 1 means perfect agreement
- κ = 0 means agreement equals chance
- κ < 0 means worse than chance (systematic disagreement)

In [ ]:
def cohens_kappa(labels_1, labels_2):
    """Compute Cohen's Kappa for two annotators.
    
    Works for any set of labels (binary, multi-class).
    """
    n = len(labels_1)
    assert n == len(labels_2), "Both annotators must label the same number of items"
    
    # Step 1: Find all unique labels
    all_labels = sorted(set(labels_1) | set(labels_2))
    
    # Step 2: Build the confusion matrix
    # confusion[i][j] = number of items where annotator 1 said label i
    #                    and annotator 2 said label j
    label_to_idx = {label: i for i, label in enumerate(all_labels)}
    k = len(all_labels)
    confusion = [[0] * k for _ in range(k)]
    
    for a, b in zip(labels_1, labels_2):
        confusion[label_to_idx[a]][label_to_idx[b]] += 1
    
    # Step 3: Observed agreement (diagonal sum / total)
    p_o = sum(confusion[i][i] for i in range(k)) / n
    
    # Step 4: Expected agreement by chance
    # For each label, multiply annotator 1's rate by annotator 2's rate
    p_e = 0
    for i in range(k):
        row_sum = sum(confusion[i][j] for j in range(k))  # Annotator 1's count for label i
        col_sum = sum(confusion[j][i] for j in range(k))  # Annotator 2's count for label i
        p_e += (row_sum / n) * (col_sum / n)
    
    # Step 5: Kappa
    if p_e == 1.0:
        return 1.0  # Both annotators always agree on one label
    kappa = (p_o - p_e) / (1 - p_e)
    
    return kappa

print("cohens_kappa() function defined.")

In [ ]:
# Test with our example from above
kappa = cohens_kappa(annotator_1, annotator_2)

print(f"Raw agreement: {raw_agreement:.0%}")
print(f"Chance agreement: {chance_agreement:.1%}")
print(f"Cohen's Kappa: {kappa:.3f}")
print()
print("Interpretation:")
if kappa < 0:
    print("  Worse than chance! Annotators systematically disagree.")
elif kappa < 0.20:
    print("  Slight agreement. Guidelines need serious revision.")
elif kappa < 0.40:
    print("  Fair agreement. Guidelines need improvement.")
elif kappa < 0.60:
    print("  Moderate agreement. Acceptable for some tasks.")
elif kappa < 0.80:
    print("  Substantial agreement. Guidelines are working well.")
else:
    print("  Almost perfect agreement.")
print()
print("Notice: 90% raw agreement but only 0.608 Kappa.")
print("The high baseline (74.5% by chance) eats most of the agreement.")

Let's build the confusion matrix to see where annotators agree and disagree.

In [ ]:
def build_confusion_matrix(labels_1, labels_2):
    """Build a confusion matrix as a 2D numpy array."""
    all_labels = sorted(set(labels_1) | set(labels_2))
    label_to_idx = {label: i for i, label in enumerate(all_labels)}
    k = len(all_labels)
    cm = np.zeros((k, k), dtype=int)
    for a, b in zip(labels_1, labels_2):
        cm[label_to_idx[a], label_to_idx[b]] += 1
    return cm, all_labels

cm, labels = build_confusion_matrix(annotator_1, annotator_2)

print("Agreement Confusion Matrix:")
print(f"{'':>15s}", end="")
for label in labels:
    name = 'Good' if label == 1 else 'Bad'
    print(f"  Ann2={name:>4s}", end="")
print()
for i, label in enumerate(labels):
    name = 'Good' if label == 1 else 'Bad'
    print(f"  Ann1={name:>4s}    ", end="")
    for j in range(len(labels)):
        print(f"  {cm[i, j]:>8d}", end="")
    print()
print()
print("Diagonal = agreement. Off-diagonal = disagreement.")
print(f"80 items: both say Good (agree)")
print(f"10 items: both say Bad (agree)")
print(f" 5 items: Ann1=Good, Ann2=Bad (disagree)")
print(f" 5 items: Ann1=Bad, Ann2=Good (disagree)")

---
## 3. Visualizing Agreement: Heatmaps

A heatmap of the confusion matrix makes disagreement patterns immediately visible. The diagonal shows agreement; everything off-diagonal is a disagreement.

In [ ]:
def plot_agreement_heatmap(labels_1, labels_2, title="Agreement Heatmap"):
    """Plot a heatmap of the agreement confusion matrix."""
    cm, label_names = build_confusion_matrix(labels_1, labels_2)
    
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm, cmap='Blues', interpolation='nearest')
    
    # Add text annotations
    for i in range(len(label_names)):
        for j in range(len(label_names)):
            color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                    fontsize=16, fontweight='bold', color=color)
    
    display_names = [str(l) for l in label_names]
    ax.set_xticks(range(len(label_names)))
    ax.set_yticks(range(len(label_names)))
    ax.set_xticklabels(display_names, fontsize=12)
    ax.set_yticklabels(display_names, fontsize=12)
    ax.set_xlabel('Annotator 2', fontsize=13)
    ax.set_ylabel('Annotator 1', fontsize=13)
    ax.set_title(title, fontsize=14)
    plt.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()

plot_agreement_heatmap(annotator_1, annotator_2,
                       title=f"Binary Agreement (\u03ba = {kappa:.3f})")

print("The dark diagonal shows most items are in agreement.")
print("But the off-diagonal cells show 10 items where annotators disagree.")

---
## 4. Multi-Class Agreement

Cohen's Kappa works for any number of categories, not just binary. Let's try a 3-category example: rating chatbot responses as "bad" (1), "okay" (2), or "good" (3).

In [ ]:
# Three categories: 1=bad, 2=okay, 3=good
# 50 items rated by two annotators
ann1_multi = [3,3,3,3,3,3,3,3,3,3,  # 10 items both think are good
              3,3,3,2,2,              # 3 good + 2 okay from ann1
              2,2,2,2,2,2,2,2,2,2,    # 10 okays from ann1
              2,2,2,3,3,              # 3 okay + 2 good from ann1
              1,1,1,1,1,1,1,1,1,1,    # 10 bads from ann1
              1,1,1,1,1,              # 5 bads from ann1
              2,2,1,1,3]              # mixed

ann2_multi = [3,3,3,3,3,3,3,3,3,3,  # 10 items both think are good
              2,2,2,2,2,              # ann2 says okay for these
              2,2,2,2,2,2,2,2,2,2,    # 10 okays agree
              2,2,2,2,2,              # ann2 says okay (some mismatch)
              1,1,1,1,1,1,1,1,1,1,    # 10 bads agree
              1,1,1,2,2,              # ann2: 3 bad + 2 okay
              3,1,2,1,2]              # mixed

kappa_multi = cohens_kappa(ann1_multi, ann2_multi)
raw_multi = sum(a == b for a, b in zip(ann1_multi, ann2_multi)) / len(ann1_multi)

print(f"Multi-class (3 categories): {len(ann1_multi)} items")
print(f"Raw agreement: {raw_multi:.0%}")
print(f"Cohen's Kappa: {kappa_multi:.3f}")

plot_agreement_heatmap(ann1_multi, ann2_multi,
                       title=f"3-Category Agreement (\u03ba = {kappa_multi:.3f})")

print("The off-diagonal cells show WHERE annotators disagree.")
print("Are they confusing adjacent categories (2 vs 3) or distant ones (1 vs 3)?")
print("Adjacent confusion suggests the scale is too fine-grained.")

---
## 5. Simulating Annotator Noise

What happens to Kappa when annotators are noisy? Let's start with a "ground truth" and add increasing levels of random noise to each annotator's labels. As noise increases, Kappa should drop.

In [ ]:
def add_noise(labels, noise_rate, rng=None):
    """Randomly flip a fraction of labels."""
    if rng is None:
        rng = np.random.RandomState(0)
    all_labels = sorted(set(labels))
    noisy = list(labels)
    for i in range(len(noisy)):
        if rng.random() < noise_rate:
            # Pick a different label at random
            other_labels = [l for l in all_labels if l != noisy[i]]
            noisy[i] = rng.choice(other_labels)
    return noisy

# Ground truth: 200 items, roughly 60% good, 40% bad
ground_truth = [1]*120 + [0]*80

noise_levels = [0.0, 0.05, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50]
kappas = []
raw_agreements = []

for noise in noise_levels:
    ann_a = add_noise(ground_truth, noise, rng=np.random.RandomState(42))
    ann_b = add_noise(ground_truth, noise, rng=np.random.RandomState(99))
    k = cohens_kappa(ann_a, ann_b)
    r = sum(a == b for a, b in zip(ann_a, ann_b)) / len(ann_a)
    kappas.append(k)
    raw_agreements.append(r)

print(f"{'Noise':>8s} | {'Raw Agree':>10s} | {'Kappa':>8s}")
print("-" * 32)
for i, noise in enumerate(noise_levels):
    print(f"{noise:>8.0%} | {raw_agreements[i]:>10.1%} | {kappas[i]:>8.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot([n*100 for n in noise_levels], raw_agreements, 's-',
        label='Raw Agreement', linewidth=2, color='#FF9800')
ax.plot([n*100 for n in noise_levels], kappas, 'o-',
        label="Cohen's Kappa", linewidth=2, color='#2196F3')

# Add threshold lines
ax.axhline(y=0.6, color='green', linestyle='--', alpha=0.5, label='\u03ba = 0.6 (substantial)')
ax.axhline(y=0.4, color='red', linestyle='--', alpha=0.5, label='\u03ba = 0.4 (fair/moderate)')

ax.set_xlabel('Noise Rate (%)', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Agreement vs Annotator Noise', fontsize=14)
ax.legend(fontsize=10)
ax.set_ylim(-0.1, 1.05)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Key observations:")
print("- Raw agreement drops slowly (it includes chance agreement)")
print("- Kappa drops faster because it strips out the chance baseline")
print("- At 30% noise, raw agreement is still ~60% but Kappa is near 0")
print("- Kappa is the honest measure of real agreement")

---
## 6. Fleiss' Kappa — Three or More Annotators

Cohen's Kappa works for exactly two annotators. In practice, we often use three or more annotators per item (for majority voting and reliability). **Fleiss' Kappa** generalizes Cohen's Kappa to any number of annotators.

In [ ]:
def fleiss_kappa(ratings_matrix):
    """Compute Fleiss' Kappa for multiple annotators.
    
    Args:
        ratings_matrix: numpy array of shape (N, k) where
            N = number of items
            k = number of categories
            ratings_matrix[i][j] = number of annotators who assigned
                                    item i to category j
    """
    N, k = ratings_matrix.shape
    n = ratings_matrix[0].sum()  # Number of annotators per item
    
    # Step 1: Compute P_i for each item (pairwise agreement within that item)
    P_items = []
    for i in range(N):
        row = ratings_matrix[i]
        P_i = (np.sum(row ** 2) - n) / (n * (n - 1))
        P_items.append(P_i)
    
    # Step 2: Average pairwise agreement
    P_bar = np.mean(P_items)
    
    # Step 3: Expected agreement by chance
    # p_j = proportion of all assignments to category j
    p_j = ratings_matrix.sum(axis=0) / (N * n)
    P_e_bar = np.sum(p_j ** 2)
    
    # Step 4: Fleiss' Kappa
    if P_e_bar == 1.0:
        return 1.0
    kappa = (P_bar - P_e_bar) / (1 - P_e_bar)
    
    return kappa

print("fleiss_kappa() function defined.")

In [ ]:
# Example: 10 items rated by 5 annotators into 3 categories (bad, okay, good)
# Each row: [count_bad, count_okay, count_good] for that item
#
# Item 1: 0 say bad, 1 says okay, 4 say good -> high agreement
# Item 5: 2 say bad, 2 say okay, 1 says good -> low agreement

ratings = np.array([
    [0, 1, 4],  # Item 1: strong agreement on good
    [0, 0, 5],  # Item 2: perfect agreement on good
    [5, 0, 0],  # Item 3: perfect agreement on bad
    [0, 2, 3],  # Item 4: mostly good
    [2, 2, 1],  # Item 5: disagreement
    [1, 4, 0],  # Item 6: mostly okay
    [0, 0, 5],  # Item 7: perfect agreement on good
    [3, 1, 1],  # Item 8: mostly bad
    [1, 1, 3],  # Item 9: mostly good
    [4, 1, 0],  # Item 10: mostly bad
])

kappa_f = fleiss_kappa(ratings)
print(f"Fleiss' Kappa for 5 annotators, 10 items, 3 categories: {kappa_f:.3f}")
print()

# Show per-item agreement
n_annotators = 5
print(f"{'Item':>6s} | {'Bad':>4s} {'Okay':>5s} {'Good':>5s} | {'Agreement':>10s}")
print("-" * 40)
for i in range(len(ratings)):
    row = ratings[i]
    # Per-item agreement: fraction of annotator pairs that agree
    p_i = (np.sum(row ** 2) - n_annotators) / (n_annotators * (n_annotators - 1))
    print(f"{i+1:>6d} | {row[0]:>4d} {row[1]:>5d} {row[2]:>5d} | {p_i:>10.2f}")

print(f"\nItems 2, 3, 7 have perfect agreement (1.00).")
print(f"Item 5 has the most disagreement (0.10).")

---
## 7. Pairwise Comparison & ELO Ratings

Instead of asking judges to rate responses on a scale, we can ask a simpler question: **"Which response is better, A or B?"** This is called pairwise comparison.

To turn many pairwise comparisons into a global ranking, we use the **ELO rating system** — the same system used in chess. Each model starts with a rating (e.g., 1500). When model A beats model B, A's rating goes up and B's goes down.

In [ ]:
def elo_expected(rating_a, rating_b):
    """Expected win probability for A against B."""
    return 1.0 / (1.0 + 10 ** ((rating_b - rating_a) / 400))

def elo_update(rating_a, rating_b, outcome_a, k=32):
    """Update ELO ratings after a match.
    
    outcome_a: 1.0 = A wins, 0.0 = B wins, 0.5 = tie
    k: update factor (higher = more sensitive to new results)
    
    Returns: (new_rating_a, new_rating_b)
    """
    expected_a = elo_expected(rating_a, rating_b)
    expected_b = 1.0 - expected_a
    outcome_b = 1.0 - outcome_a
    
    new_a = rating_a + k * (outcome_a - expected_a)
    new_b = rating_b + k * (outcome_b - expected_b)
    
    return new_a, new_b

# Quick test: two equally rated models, A wins
r_a, r_b = elo_update(1500, 1500, outcome_a=1.0)
print(f"Before: A=1500, B=1500")
print(f"A wins.")
print(f"After:  A={r_a:.0f}, B={r_b:.0f}")
print()

# Upset: weaker model beats stronger
r_a2, r_b2 = elo_update(1400, 1600, outcome_a=1.0)
print(f"Before: A=1400 (underdog), B=1600 (favorite)")
print(f"A wins (upset)!")
print(f"After:  A={r_a2:.0f}, B={r_b2:.0f}")
print("Upsets cause bigger rating swings.")

Now let's simulate a full tournament between 4 chatbot models with different quality levels.

In [ ]:
# Simulate: 4 models with different "true" quality
# Higher quality = more likely to win a pairwise comparison
models = {
    'ModelA': {'true_quality': 0.9, 'rating': 1500},
    'ModelB': {'true_quality': 0.7, 'rating': 1500},
    'ModelC': {'true_quality': 0.5, 'rating': 1500},
    'ModelD': {'true_quality': 0.3, 'rating': 1500},
}

model_names = list(models.keys())
rng = np.random.RandomState(42)

# Track rating history for plotting
history = {name: [1500] for name in model_names}

# Run 500 random matchups
n_matches = 500
for _ in range(n_matches):
    # Pick two random models
    i, j = rng.choice(len(model_names), size=2, replace=False)
    name_a, name_b = model_names[i], model_names[j]
    
    # Simulate who wins based on true quality difference
    q_a = models[name_a]['true_quality']
    q_b = models[name_b]['true_quality']
    # Win probability proportional to quality difference
    p_a_wins = q_a / (q_a + q_b)
    outcome = 1.0 if rng.random() < p_a_wins else 0.0
    
    # Update ELO
    new_a, new_b = elo_update(
        models[name_a]['rating'],
        models[name_b]['rating'],
        outcome,
        k=16  # Smaller k for more stable ratings
    )
    models[name_a]['rating'] = new_a
    models[name_b]['rating'] = new_b
    
    for name in model_names:
        history[name].append(models[name]['rating'])

# Final rankings
print("Final ELO Rankings (after 500 matchups):")
print(f"{'Model':>10s} | {'True Quality':>13s} | {'ELO Rating':>11s}")
print("-" * 40)
sorted_models = sorted(models.items(), key=lambda x: x[1]['rating'], reverse=True)
for name, info in sorted_models:
    print(f"{name:>10s} | {info['true_quality']:>13.1f} | {info['rating']:>11.0f}")

print("\nELO correctly ranks the models by their true quality!")

In [ ]:
# Plot ELO convergence
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#4CAF50', '#2196F3', '#FF9800', '#F44336']
for i, name in enumerate(model_names):
    ax.plot(history[name], label=f"{name} (q={models[name]['true_quality']})",
            linewidth=2, color=colors[i], alpha=0.8)

ax.set_xlabel('Number of Matchups', fontsize=12)
ax.set_ylabel('ELO Rating', fontsize=12)
ax.set_title('ELO Rating Convergence Over 500 Matchups', fontsize=14)
ax.legend(fontsize=10, loc='center right')
ax.grid(True, alpha=0.3)
ax.axhline(y=1500, color='gray', linestyle=':', alpha=0.3)
plt.tight_layout()
plt.show()

print("Key observations:")
print("- All models start at 1500 and diverge based on match outcomes")
print("- Ratings stabilize after ~200 matchups")
print("- The final ranking matches the true quality order")
print("- This is how Chatbot Arena ranks LLMs using human preferences")

---
## 8. Measuring Position Bias in Pairwise Comparison

One danger in A/B testing: judges tend to prefer whichever response they see first. This is called **position bias**. Let's simulate it and see how it distorts results.

In [ ]:
# Simulate pairwise comparison with and without position bias
n_comparisons = 1000
rng = np.random.RandomState(42)

# Two models of EQUAL quality
true_quality_a = 0.5
true_quality_b = 0.5

# Without position bias: fair coin flip (equal quality = 50/50)
wins_fair = rng.binomial(1, 0.5, size=n_comparisons)
win_rate_fair = wins_fair.mean()

# With position bias: first model gets +10% boost
# A is shown first in all comparisons
position_boost = 0.10
wins_biased = rng.binomial(1, 0.5 + position_boost, size=n_comparisons)
win_rate_biased = wins_biased.mean()

# With randomized order: half the time A is first, half the time B is first
order = rng.binomial(1, 0.5, size=n_comparisons)  # 1 = A first, 0 = B first
wins_randomized = []
for i in range(n_comparisons):
    if order[i] == 1:  # A shown first
        win = rng.binomial(1, 0.5 + position_boost)
    else:  # B shown first (B gets the boost, so A's win prob drops)
        win = rng.binomial(1, 0.5 - position_boost)
    wins_randomized.append(win)
win_rate_randomized = np.mean(wins_randomized)

print("Two EQUAL quality models (true win rate should be 50%):")
print(f"  No bias:          A wins {win_rate_fair:.1%}")
print(f"  A always first:   A wins {win_rate_biased:.1%}  <- biased!")
print(f"  Randomized order: A wins {win_rate_randomized:.1%}  <- bias cancels out")
print()
print("Lesson: ALWAYS randomize presentation order.")
print("Without randomization, position bias can make equal models")
print("look very different.")

---
## 9. Putting It All Together: A Mini Evaluation Study

Let's simulate a complete human evaluation study with 3 annotators rating 20 chatbot responses on a 3-point scale (bad, okay, good). We will compute both raw agreement and Fleiss' Kappa, then visualize the results.

In [ ]:
# Simulate 3 annotators rating 20 items on 3 categories
# True quality is hidden — annotators see it with noise
rng = np.random.RandomState(42)

n_items = 20
n_annotators = 3
categories = ['bad', 'okay', 'good']

# True quality of each item (0=bad, 1=okay, 2=good)
true_quality = rng.choice([0, 1, 2], size=n_items, p=[0.2, 0.4, 0.4])

# Each annotator sees the truth + some noise
noise_rate = 0.15  # 15% chance of flipping to a different category
all_annotations = []
for ann_id in range(n_annotators):
    annotations = []
    for item in range(n_items):
        if rng.random() < noise_rate:
            # Noisy: pick a different category
            other = [c for c in range(3) if c != true_quality[item]]
            annotations.append(rng.choice(other))
        else:
            annotations.append(true_quality[item])
    all_annotations.append(annotations)

# Build Fleiss ratings matrix
ratings_matrix = np.zeros((n_items, 3), dtype=int)
for ann_id in range(n_annotators):
    for item in range(n_items):
        ratings_matrix[item, all_annotations[ann_id][item]] += 1

# Compute Fleiss' Kappa
kappa_study = fleiss_kappa(ratings_matrix)

# Compute pairwise Cohen's Kappa for each annotator pair
print("=" * 50)
print("Mini Evaluation Study: 3 annotators, 20 items, 3 categories")
print("=" * 50)
print(f"\nFleiss' Kappa (all 3 annotators): {kappa_study:.3f}")
print()

print("Pairwise Cohen's Kappa:")
for i in range(n_annotators):
    for j in range(i+1, n_annotators):
        k_pair = cohens_kappa(all_annotations[i], all_annotations[j])
        raw = sum(a == b for a, b in zip(all_annotations[i], all_annotations[j])) / n_items
        print(f"  Annotator {i+1} vs {j+1}: \u03ba = {k_pair:.3f} (raw = {raw:.0%})")

# Show the annotations
print(f"\n{'Item':>6s} | {'True':>5s} | {'Ann1':>5s} {'Ann2':>5s} {'Ann3':>5s} | {'Agree?':>7s}")
print("-" * 50)
for item in range(n_items):
    t = categories[true_quality[item]]
    a1 = categories[all_annotations[0][item]]
    a2 = categories[all_annotations[1][item]]
    a3 = categories[all_annotations[2][item]]
    all_same = a1 == a2 == a3
    print(f"{item+1:>6d} | {t:>5s} | {a1:>5s} {a2:>5s} {a3:>5s} | {'YES' if all_same else 'no':>7s}")

In [ ]:
# Visualize: per-item agreement as a bar chart
item_agreement = []
for item in range(n_items):
    row = ratings_matrix[item]
    p_i = (np.sum(row ** 2) - n_annotators) / (n_annotators * (n_annotators - 1))
    item_agreement.append(p_i)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Per-item agreement
colors_bar = ['#F44336' if a < 0.5 else '#FF9800' if a < 1.0 else '#4CAF50'
              for a in item_agreement]
axes[0].bar(range(1, n_items+1), item_agreement, color=colors_bar)
axes[0].axhline(y=1.0, color='green', linestyle='--', alpha=0.3, label='Perfect')
axes[0].axhline(y=0.33, color='red', linestyle='--', alpha=0.3, label='Chance (3 cats)')
axes[0].set_xlabel('Item Number', fontsize=12)
axes[0].set_ylabel('Pairwise Agreement', fontsize=12)
axes[0].set_title('Per-Item Annotator Agreement', fontsize=13)
axes[0].legend(fontsize=9)
axes[0].set_ylim(0, 1.1)

# Plot 2: Category distribution across all annotations
cat_totals = ratings_matrix.sum(axis=0)
axes[1].bar(categories, cat_totals, color=['#F44336', '#FF9800', '#4CAF50'])
axes[1].set_xlabel('Category', fontsize=12)
axes[1].set_ylabel('Total Annotations', fontsize=12)
axes[1].set_title('Category Distribution Across All Annotators', fontsize=13)
for i, v in enumerate(cat_totals):
    axes[1].text(i, v + 0.5, str(v), ha='center', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

n_perfect = sum(1 for a in item_agreement if a == 1.0)
n_disagree = sum(1 for a in item_agreement if a < 0.5)
print(f"Items with perfect agreement: {n_perfect}/{n_items}")
print(f"Items with low agreement (<0.5): {n_disagree}/{n_items}")
print(f"Overall Fleiss' \u03ba = {kappa_study:.3f}")

---
## Summary

In this notebook you built:

- **Cohen's Kappa from scratch** — measures agreement between two annotators, correcting for the chance baseline
- **Fleiss' Kappa from scratch** — extends to three or more annotators
- **Annotator noise simulation** — showed how Kappa drops faster than raw agreement as noise increases
- **Agreement heatmaps** — visualize where annotators agree and disagree
- **ELO rating system** — turns pairwise comparisons into a global ranking (just like Chatbot Arena)
- **Position bias simulation** — demonstrated why randomizing presentation order is critical
- **A mini evaluation study** — combining everything into a realistic workflow

For the full math (Krippendorff's Alpha, sample size formulas, power analysis) and staff-level interview questions, see the [interview deep-dive](./human-evaluation-interview.md).

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)